# ATP Tennis Match Prediction — Data Preparation
Preprocessing raw ATP match data (2018–2024), engineering historical features without data leakage, and splitting into train/test sets.

## 1. Load Match Datasets

In [1]:
import glob
import os
import pandas as pd
import numpy as np

files = sorted(glob.glob("../data/raw/atp_matches_*.csv"))
if not files:
    files = sorted(glob.glob("data/raw/atp_matches_*.csv"))

dfs = []
for f in files:
    df = pd.read_csv(f)
    df["year"] = int(os.path.basename(f).split("_")[-1].replace(".csv", ""))
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
print("Combined shape:", data.shape)
data.head()

Combined shape: (18877, 50)


,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points,year
0,2018-M020,Brisbane,Hard,32,A,20180101,271,105992,NaN,NaN,...,33.0,19.0,14.0,1.0,4.0,47.0,1010.0,52.0,909.0,2018
1,2018-M020,Brisbane,Hard,32,A,20180101,272,111577,NaN,NaN,...,25.0,7.0,9.0,7.0,11.0,54.0,890.0,94.0,593.0,2018
2,2018-M020,Brisbane,Hard,32,A,20180101,273,104797,NaN,NaN,...,37.0,29.0,15.0,10.0,16.0,63.0,809.0,30.0,1391.0,2018
3,2018-M020,Brisbane,Hard,32,A,20180101,275,200282,NaN,WC,...,33.0,17.0,11.0,4.0,6.0,208.0,245.0,44.0,1055.0,2018
4,2018-M020,Brisbane,Hard,32,A,20180101,276,111581,NaN,Q,...,28.0,5.0,9.0,0.0,2.0,175.0,299.0,68.0,755.0,2018


## 2. Data Cleaning

In [2]:
data = data.drop_duplicates().reset_index(drop=True)
data.columns = data.columns.str.strip().str.lower()

numeric_cols = [
    "winner_rank", "loser_rank",
    "winner_rank_points", "loser_rank_points",
    "winner_age", "loser_age"
]
for col in numeric_cols:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors="coerce")

categorical_cols = ["surface", "round", "tourney_name", "winner_name", "loser_name"]
for col in categorical_cols:
    if col in data.columns:
        data[col] = data[col].astype("string").str.strip()

required_cols = [
    "winner_id", "loser_id",
    "winner_rank", "loser_rank",
    "winner_rank_points", "loser_rank_points",
    "surface"
]
data = data.dropna(subset=required_cols).reset_index(drop=True)

data["winner_age"] = data["winner_age"].fillna(data["winner_age"].median())
data["loser_age"] = data["loser_age"].fillna(data["loser_age"].median())

print("Cleaned shape:", data.shape)
data[required_cols].isnull().sum()

Cleaned shape: (18572, 50)


winner_id             0
loser_id              0
winner_rank           0
loser_rank            0
winner_rank_points    0
loser_rank_points     0
surface               0
dtype: int64

## 3. Historical Feature Engineering
Calculates pre-match win rate, surface win rate, and head-to-head records. Statistics are updated strictly after each match to avoid lookahead bias.

In [3]:
history_data = data.sort_values(by=["year", "tourney_date"]).reset_index(drop=True)

player_stats = {}
h2h = {}
historical_rows = []

def get_player_stats(player_id):
    if player_id not in player_stats:
        player_stats[player_id] = {
            "matches": 0,
            "wins": 0,
            "surface_matches": {},
            "surface_wins": {}
        }
    return player_stats[player_id]

def get_h2h(player1, player2):
    key = tuple(sorted([player1, player2]))
    if key not in h2h:
        h2h[key] = {player1: 0, player2: 0}
    return h2h[key].get(player1, 0)

for _, row in history_data.iterrows():
    winner = row["winner_id"]
    loser = row["loser_id"]
    surface = row["surface"]

    if pd.isna(winner) or pd.isna(loser):
        continue

    w = get_player_stats(winner)
    l = get_player_stats(loser)

    w_win_rate = w["wins"] / w["matches"] if w["matches"] else 0.5
    l_win_rate = l["wins"] / l["matches"] if l["matches"] else 0.5

    w_sm = w["surface_matches"].get(surface, 0)
    w_sw = w["surface_wins"].get(surface, 0)
    l_sm = l["surface_matches"].get(surface, 0)
    l_sw = l["surface_wins"].get(surface, 0)

    w_surface_rate = w_sw / w_sm if w_sm else 0.5
    l_surface_rate = l_sw / l_sm if l_sm else 0.5

    w_h2h = get_h2h(winner, loser)
    l_h2h = get_h2h(loser, winner)

    # winner perspective
    historical_rows.append({
        "player1": winner,
        "player2": loser,
        "surface": surface,
        "rank1": row["winner_rank"],
        "rank2": row["loser_rank"],
        "points1": row["winner_rank_points"],
        "points2": row["loser_rank_points"],
        "age1": row["winner_age"],
        "age2": row["loser_age"],
        "winrate1": w_win_rate,
        "winrate2": l_win_rate,
        "surfacerate1": w_surface_rate,
        "surfacerate2": l_surface_rate,
        "h2h1": w_h2h,
        "h2h2": l_h2h,
        "target": 1,
        "year": row["year"]
    })

    # loser perspective
    historical_rows.append({
        "player1": loser,
        "player2": winner,
        "surface": surface,
        "rank1": row["loser_rank"],
        "rank2": row["winner_rank"],
        "points1": row["loser_rank_points"],
        "points2": row["winner_rank_points"],
        "age1": row["loser_age"],
        "age2": row["winner_age"],
        "winrate1": l_win_rate,
        "winrate2": w_win_rate,
        "surfacerate1": l_surface_rate,
        "surfacerate2": w_surface_rate,
        "h2h1": l_h2h,
        "h2h2": w_h2h,
        "target": 0,
        "year": row["year"]
    })

    # update stats after match
    w["matches"] += 1
    w["wins"] += 1
    l["matches"] += 1
    w["surface_matches"][surface] = w["surface_matches"].get(surface, 0) + 1
    w["surface_wins"][surface] = w["surface_wins"].get(surface, 0) + 1
    l["surface_matches"][surface] = l["surface_matches"].get(surface, 0) + 1

    key = tuple(sorted([winner, loser]))
    if key not in h2h:
        h2h[key] = {winner: 0, loser: 0}
    h2h[key][winner] = h2h[key].get(winner, 0) + 1

historical_data = pd.DataFrame(historical_rows)
print("Historical dataset:", historical_data.shape)
historical_data.head()

Historical dataset: (37144, 17)


,player1,player2,surface,rank1,rank2,points1,points2,age1,age2,winrate1,winrate2,surfacerate1,surfacerate2,h2h1,h2h2,target,year
0,105992,104919,Hard,47.0,52.0,1010.0,909.0,25.6,30.6,0.5,0.5,0.5,0.5,0,0,1,2018
1,104919,105992,Hard,52.0,47.0,909.0,1010.0,30.6,25.6,0.5,0.5,0.5,0.5,0,0,0,2018
2,111577,111442,Hard,54.0,94.0,890.0,593.0,21.2,23.7,0.5,0.5,0.5,0.5,0,0,1,2018
3,111442,111577,Hard,94.0,54.0,593.0,890.0,23.7,21.2,0.5,0.5,0.5,0.5,0,0,0,2018
4,104797,106000,Hard,63.0,30.0,809.0,1391.0,31.3,25.6,0.5,0.5,0.5,0.5,0,0,1,2018


## 4. Differential Features

In [4]:
improved_data = pd.DataFrame()

improved_data["rank_diff"] = historical_data["rank1"] - historical_data["rank2"]
improved_data["rank_points_diff"] = historical_data["points1"] - historical_data["points2"]
improved_data["age_diff"] = historical_data["age1"] - historical_data["age2"]
improved_data["win_rate_diff"] = historical_data["winrate1"] - historical_data["winrate2"]
improved_data["surface_rate_diff"] = historical_data["surfacerate1"] - historical_data["surfacerate2"]
improved_data["h2h_diff"] = historical_data["h2h1"] - historical_data["h2h2"]
improved_data["surface"] = historical_data["surface"]
improved_data["year"] = historical_data["year"]
improved_data["target"] = historical_data["target"]

numeric_features = [
    "rank_diff", "rank_points_diff", "age_diff",
    "win_rate_diff", "surface_rate_diff", "h2h_diff"
]
for col in numeric_features:
    improved_data[col] = improved_data[col].fillna(improved_data[col].median())

print("Final dataset shape:", improved_data.shape)
improved_data.head()

Final dataset shape: (37144, 9)


,rank_diff,rank_points_diff,age_diff,win_rate_diff,surface_rate_diff,h2h_diff,surface,year,target
0,-5.0,101.0,-5.0,0.0,0.0,0,Hard,2018,1
1,5.0,-101.0,5.0,0.0,0.0,0,Hard,2018,0
2,-40.0,297.0,-2.5,0.0,0.0,0,Hard,2018,1
3,40.0,-297.0,2.5,0.0,0.0,0,Hard,2018,0
4,33.0,-582.0,5.7,0.0,0.0,0,Hard,2018,1


## 5. Chronological Train / Test Split

In [5]:
train_data = improved_data[improved_data["year"] < 2024].copy()
test_data = improved_data[improved_data["year"] == 2024].copy()

print("Training samples (2018-2023):", len(train_data))
print("Testing samples (2024):      ", len(test_data))

out_dir = "../data" if os.path.exists("../data") else "data"
os.makedirs(out_dir, exist_ok=True)

train_data.to_csv(os.path.join(out_dir, "train_data.csv"), index=False)
test_data.to_csv(os.path.join(out_dir, "test_data.csv"), index=False)
data.to_csv(os.path.join(out_dir, "cleaned_atp_data.csv"), index=False)

print(f"Saved train_data.csv, test_data.csv, and cleaned_atp_data.csv to {out_dir}/")

Training samples (2018-2023): 31084
Testing samples (2024):       6060


Saved train_data.csv, test_data.csv, and cleaned_atp_data.csv to ../data/
